# LogiTrack: Notebook 05: SQL Business Analysis (Step 10, SQL part)
Runs the 10 business queries in `sql/analysis_queries.sql` against the database through the Python-SQL connection (SQLAlchemy). The same file feeds the Streamlit dashboard, so the notebook and the dashboard always show identical logic.

Techniques used: SELECT/WHERE/ORDER BY · GROUP BY/HAVING · INNER and LEFT JOIN · subqueries · CTEs · window functions (RANK, LAG, SUM OVER) · aggregations.

In [1]:
# --- Setup: make the project root importable and load shared helpers -------------
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

from src import utils
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="Set2")
%matplotlib inline
import re
engine = utils.get_engine()
raw_sql = (utils.SQL_DIR / "analysis_queries.sql").read_text()
queries = {}
for block in re.split(r"^-- name: ", raw_sql, flags=re.M)[1:]:
    header, body = block.split("\n", 1)
    qid, title = [p.strip() for p in header.split("|", 1)]
    queries[qid] = (title, body.split("\n-- ====")[0].strip().rstrip(";"))
print(engine.url.render_as_string(hide_password=True), "|", len(queries), "queries loaded")

sqlite:////workspace/LogiTrack/data/logitrack.db | 18 queries loaded


In [2]:
def show(qid):
    title, sql = queries[qid]
    print(title); print("-" * len(title))
    df = utils.run_query(sql, engine=engine)
    display(df)
    return df

In [3]:
q1 = show('q01')

1. Overall delivery performance
-------------------------------


,total_shipments,total_deliveries,delivered,delayed,in_transit,cancelled,success_rate_pct,on_time_pct_of_completed,avg_delay_hours,avg_rating,avg_delivery_days
0,3000,2712,2232,480,147,141,74.4,28.58,7.08,3.89,1.11


**Insight:** 74.4% success (Delivered), 16% Delayed and 9.6% not completed. Only 28.6% of completed deliveries have zero delay, so most 'Delivered' orders still run slightly late.

In [4]:
q2 = show('q02')

2. Average delay by route type
------------------------------


,route_type,shipments,completed_deliveries,avg_delay_hours,max_delay_hours,delayed_status_pct,avg_distance_km
0,Express,597,554,7.40,47.92,16.58,449.0
1,Economy,643,571,7.32,47.89,16.80,1663.0
2,Standard,1760,1587,6.88,47.97,15.51,1231.0


**Insight:** route *type* makes little difference (6.9-7.4 h). Express routes are not faster to arrive *on promise*, which points to an SLA-setting problem for Express.

In [5]:
q3 = show('q03')

3. Top performing drivers
-------------------------


,driver_id,driver_name,driver_experience,experience_level,driver_rating,shipments,avg_customer_rating,avg_delay_hours,success_rate_pct,rating_rank
0,15,Kiran Pillai,15,Expert,4.13,33,4.33,4.65,87.9,1
1,75,Kiran Kumar,3,Junior,4.16,35,4.26,5.52,88.6,2
2,20,Varun Iyer,16,Expert,4.56,55,4.25,7.42,83.6,3
3,5,Amit Kumar,15,Expert,4.69,32,4.16,3.93,93.8,4
4,70,Shyam Menon,2,Junior,4.02,41,4.15,6.76,78.0,5
5,66,Tarun Shah,12,Senior,3.97,40,4.13,6.45,82.5,6
6,35,Amit Patel,5,Mid,3.99,34,4.12,2.70,76.5,7
7,60,Rajesh Sharma,12,Senior,4.06,26,4.12,6.19,84.6,8
8,67,Vikram Shah,6,Mid,4.03,31,4.10,4.24,87.1,9
9,58,Suresh Menon,15,Expert,4.52,24,4.08,5.55,79.2,10


**Insight:** top-rated drivers come from every experience band (two Juniors are in the top 10). Recognition should be based on performance, not tenure.

In [6]:
q4 = show('q04')

4. Delivery performance by warehouse
------------------------------------


,warehouse_id,warehouse_name,city,state,capacity,shipments,success_rate_pct,avg_delay_hours,avg_rating,success_rank
0,2,Warehouse_2,Delhi,Delhi,3860,218,77.98,6.51,4.02,1
1,15,Warehouse_15,Patna,Bihar,5433,194,77.84,6.13,3.80,2
2,10,Warehouse_10,Lucknow,Uttar Pradesh,14363,166,77.71,6.32,3.88,3
3,11,Warehouse_11,Chandigarh,Punjab,19023,227,77.53,6.41,3.97,4
4,1,Warehouse_1,Mumbai,Maharashtra,18795,204,76.47,7.00,3.96,5
5,5,Warehouse_5,Chennai,Tamil Nadu,14284,196,75.51,7.62,3.95,6
6,3,Warehouse_3,Bangalore,Karnataka,8390,195,75.38,7.17,3.87,7
7,13,Warehouse_13,Indore,Madhya Pradesh,4685,199,74.37,6.24,3.86,8
8,7,Warehouse_7,Pune,Maharashtra,19850,201,74.13,6.56,3.98,9
9,12,Warehouse_12,Kochi,Kerala,11322,210,73.33,7.94,3.93,10


**Insight:** Delhi, Patna, Lucknow and Chandigarh lead at about 78% success. Jaipur (67%) and Nagpur (69%) trail and also have the highest average delays (8.3-8.5 h).

In [7]:
q5 = show('q05')

5. Monthly delivery trends
--------------------------


,month,shipments,delivered,delayed,success_rate_pct,avg_delay_hours,avg_rating,delay_change_vs_prev_month,cumulative_shipments
0,2024-01,118,88,16,74.58,6.23,3.86,NaN,118
1,2024-02,122,87,20,71.31,6.66,3.80,0.43,240
2,2024-03,116,75,25,64.66,9.18,3.72,2.52,356
3,2024-04,146,111,22,76.03,7.24,3.98,-1.94,502
4,2024-05,146,118,18,80.82,6.64,4.02,-0.60,648
5,2024-06,117,87,19,74.36,6.22,3.85,-0.42,765
6,2024-07,128,94,25,73.44,7.69,3.87,1.47,893
7,2024-08,119,88,20,73.95,7.14,3.90,-0.55,1012
8,2024-09,100,77,15,77.00,6.99,3.95,-0.15,1112
9,2024-10,143,105,26,73.43,6.89,3.93,-0.10,1255


**Insight:** volume is steady at about 100-146 shipments a month. Delay spikes in Mar-2024 (9.2 h), Dec-2024 (8.8 h) and Apr-May 2025 (8.3-8.8 h) are matched by success-rate dips (63-68%).

In [8]:
q6 = show('q06')

6. Vehicle utilization
----------------------


,vehicle_type,vehicles,active_vehicles,idle_vehicles,shipments,shipments_per_vehicle,total_weight_kg,avg_capacity_utilization_pct,avg_delay_hours
0,Bike,11,10,1,718,65.3,24057.0,53.78,6.45
1,Van,13,11,2,823,63.3,385787.0,46.05,6.81
2,Truck,21,20,1,1459,69.5,1938888.0,45.00,7.52


**Insight:** bikes carry the highest load factor (54%). Trucks run at only 45% of capacity even though they move about 82% of total weight. 4 of the 45 vehicles are idle (maintenance/retired).

In [9]:
q7 = show('q07')

7. Rating by delivery status
----------------------------


,delivery_status,feedback_count,avg_rating,positive_pct,negative_pct,share_of_shipments_pct
0,Delivered,2232,4.35,90.5,0.0,74.4
1,Delayed,480,2.91,20.2,29.4,16.0
2,In-Transit,147,2.02,0.0,71.4,4.9
3,Cancelled,141,1.91,0.0,73.0,4.7


**Insight:** Delivered orders average 4.35 stars against 2.91 for Delayed and about 2 for In-Transit/Cancelled. 71-73% of failed orders get a negative rating.

In [10]:
q8 = show('q08')

8. Delay analysis by distance category
--------------------------------------


,distance_category,shipments,min_km,max_km,avg_delay_hours,delayed_pct_of_completed,avg_transit_hours,avg_rating
0,Short,546,86.0,496.0,7.64,73.59,5.9,3.84
1,Medium,784,520.0,974.0,6.98,72.45,17.2,3.93
2,Long,705,1003.0,1495.0,7.22,69.63,28.8,3.89
3,Very Long,965,1510.0,2494.0,6.74,70.66,45.0,3.89


**Insight:** delay does *not* grow with distance (Short 7.6 h vs Very Long 6.7 h), so distance is not the bottleneck.

In [11]:
q9 = show('q09')

9. Driver performance by experience level
-----------------------------------------


,experience_level,drivers,avg_experience_years,shipments,avg_shipments_per_driver,avg_delay_hours,avg_success_rate_pct,avg_customer_rating
0,Junior,15,1.7,608,40.5,7.64,72.97,3.86
1,Mid,18,5.8,716,39.8,7.13,74.45,3.87
2,Senior,21,10.8,821,39.1,6.77,74.71,3.90
3,Expert,21,16.7,855,40.7,6.67,75.44,3.94


**Insight:** average delay falls steadily from Junior (7.6 h) to Expert (6.7 h), and success rises from 73.0% to 75.4%. The gain from experience is real but small.

In [12]:
q10 = show('q10')

10. Day of week analysis
------------------------


,day_of_week,shipments,share_pct,avg_delay_hours,success_rate_pct,avg_rating
0,Monday,399,13.3,8.09,71.93,3.88
1,Tuesday,412,13.7,7.15,72.57,3.84
2,Wednesday,415,13.8,6.72,76.39,3.93
3,Thursday,440,14.7,7.22,75.45,3.93
4,Friday,462,15.4,6.78,75.76,3.88
5,Saturday,455,15.2,6.91,75.38,3.89
6,Sunday,417,13.9,6.76,72.90,3.90


**Insight:** Monday dispatches have the highest delay (8.1 h) and the lowest success (71.9%), consistent with a weekend backlog.

In [13]:
# Supporting dashboard queries also run cleanly
for qid in [k for k in queries if k.startswith("s")]:
    print(f"{qid}: {queries[qid][0]:<55} rows = {len(utils.run_query(queries[qid][1], engine=engine))}")

s01: Delivery status distribution                            rows = 4
s02: Route performance comparison (lanes)                    rows = 150
s03: Driver workload distribution                            rows = 75
s04: Distance vs delay (shipment level, completed deliveries) rows = 2712
s05: Rating distribution                                     rows = 5
s06: Rating trends over time                                 rows = 24
s07: Rating by delay severity                                rows = 4
s08: Delay by warehouse and quarter (heatmap)                rows = 60
